# HEADING

In [1]:
from haystack import Pipeline, Document, component
from milvus_haystack import MilvusDocumentStore
from haystack.components.converters import PyPDFToDocument
from haystack.components.preprocessors import DocumentCleaner, DocumentSplitter
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.writers import DocumentWriter
from haystack.utils import Secret
from pymilvus import MilvusClient, DataType
from typing import List
from pymilvus import MilvusClient, DataType
import os
from openai import AsyncOpenAI
import asyncio
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [36]:
client = MilvusClient(
    uri=os.getenv("ZILLIZ_CLUSTER_ENDPOINT"),
    token=os.getenv("ZILLIZ_CLUSTER_TOKEN")
)

In [37]:
col_name = client.list_collections()[0]
print(col_name)
client.load_collection(col_name)
client.get_load_state(col_name)

TaggingTestCollection


{'state': <LoadState: Loaded>}

In [39]:
filter = 'metadata["tags"] == []'
# filter = ""
# filter = 'metadata["page_number"] == 6'

res = client.query(
    collection_name=col_name,
    filter=filter,
    # output_fields=["id", "text"],
    output_fields=["*"],
    # limit=2
)

print(res)
print(len(res))

data: ["{'metadata': {'tags': []}, 'text': 'Lebron James is the career points leader in the NBA.', 'id': 'king_james', 'vector': [np.float32(0.1), np.float32(0.1)]}", "{'metadata': {'tags': []}, 'text': 'Stephen Curry is the only player in NBA history to be unanimously voted as the regular season MVP.', 'id': 'night_night', 'vector': [np.float32(0.1), np.float32(0.1)]}"]
2


In [46]:
def dict_to_doc(d: dict) -> Document:
    return Document(id=d["id"], embedding=d["vector"], content=d["text"], meta={"metadata": d["metadata"]})


In [47]:
res = client.query(
    collection_name=col_name,
    filter=filter,
    output_fields=["*"],
)

# docs = [Document.from_dict(d) for d in res] # this doesn't work
# res[0]["metadata"]
docs = [dict_to_doc(d) for d in res]
docs[0]

Document(id=king_james, content: 'Lebron James is the career points leader in the NBA.', meta: {'metadata': {'tags': []}}, embedding: vector of size 2)

In [ ]:
# @component
# class LLMTagger:
#     def __init__(self, prompt: str):
#         self.prompt_template = prompt

#     @component.output_types(documents=List[Document])
#     def run(self, documents: List[Document]):
#         docs = [self._add_metadata(doc) for doc in documents]
#         return {"documents": documents}
    
#     async def tag(self, documents: List[Document]):
#         tasks = []
#         asyncio.gather(*tasks)
#         asyncio.run()

In [ ]:
res = client.get(
    collection_name=col_name,
    ids=["00d10142bdb9d200125ec87d0a2dab835cc4f23a89fa318ee0acbb187fb9326d",
         "049ca25d41fdad9880c104633164071b5321a320f1791c657a7aa4375a651b19"],
    # output_fields=["id", "text"]
    output_fields = ["*"]
)

print(len(res))
print(res)

# Custom components
## MilvusQueryRetriever

In [2]:
@component
class MilvusQueryRetriever:
    def __init__(self):
        self.client = MilvusClient(
            uri=Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
            token=Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value()
        )

    @component.output_types(documents=List[Document])
    def run(self, collection_name: str, ids: list[str] | None=None):
        if ids:
            # Retrieve specific IDs
            res = self.client.get(
                collection_name=collection_name,
                ids=ids,
                output_fields = ["*"]
            )
        else:
            # If no IDs provided, query all entities with empty tags
            res = self.client.query(
                collection_name=collection_name,
                filter='metadata["tags"] == []',
                output_fields=["*"],
            )

        # Convert from entity format to Document format
        return {"documents": [self.dict_to_doc(d) for d in res]}
    
    def dict_to_doc(self, d: dict) -> Document:
        return Document(id=d["id"], embedding=d["vector"], content=d["text"], meta={"metadata": d["metadata"]})

In [17]:
query_retriever = MilvusQueryRetriever()

In [18]:
col_name = "TaggingTestCollection"
docs = query_retriever.run(collection_name=col_name)
print(len(docs))
docs[0]

2


Document(id=king_james, content: 'Lebron James is the career points leader in the NBA.', meta: {'metadata': {'tags': []}}, embedding: vector of size 2)

In [19]:
ids = ["night_night"]
docs = query_retriever.run(collection_name=col_name, ids=ids)
print(len(docs))
docs[0]

1


Document(id=night_night, content: 'Stephen Curry is the only player in NBA history to be unanimously voted as the regular season MVP.', meta: {'metadata': {'tags': []}}, embedding: vector of size 2)

## SynchronousLLMTagger

In [ ]:
from haystack.components.builders import PromptBuilder
from haystack.components.generators import OpenAIGenerator

prompt_template = """Given the context inside the triple backticks below, tag the topic. You may tag multiple topics.
                     Please format the tags as strings in a Python list, e.g. ["current events", "USA"].
                     Return only the list; return no other text.

                     Context:
                     ```
                     {{doc.content}}
                     ```

                     Tags as a Python list of strings:
                  """

pipe = Pipeline()
pipe.add_component("prompt_builder", PromptBuilder(template=prompt_template))
# pipe.add_component("generator", OpenAIGenerator(model="gpt-3.5-turbo", generation_kwargs={"temperature": 0.7, "max_tokens": 500}))
pipe.add_component("generator", OpenAIGenerator(generation_kwargs={"temperature": 0.7, "max_tokens": 500}))
pipe.connect("prompt_builder", "generator")

🚅 Components
  - prompt_builder: PromptBuilder
  - generator: OpenAIGenerator
🛤️ Connections
  - prompt_builder.prompt -> generator.prompt (str)

In [8]:
doc = Document(content="Lebron James is the career points leader in the NBA.")

tag = pipe.run({"prompt_builder": {"doc": doc}})
tag

{'generator': {'replies': ['["sports", "NBA", "LeBron James", "basketball"]'],
  'meta': [{'model': 'gpt-4o-mini-2024-07-18',
    'index': 0,
    'finish_reason': 'stop',
    'usage': {'completion_tokens': 16,
     'prompt_tokens': 92,
     'total_tokens': 108,
     'completion_tokens_details': CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0),
     'prompt_tokens_details': PromptTokensDetails(audio_tokens=0, cached_tokens=0)}}]}}

In [18]:
tag["generator"]["replies"][0]

'["sports", "NBA", "LeBron James", "basketball"]'

In [19]:
import json

json.loads(tag["generator"]["replies"][0])

['sports', 'NBA', 'LeBron James', 'basketball']

In [3]:
from haystack.components.builders import PromptBuilder
from haystack.components.generators import OpenAIGenerator
import json

@component
class SyncLLMTagger:
    def __init__(self, prompt_template: str):
        pipe = Pipeline()
        pipe.add_component("prompt_builder", PromptBuilder(template=prompt_template))
        pipe.add_component("generator", OpenAIGenerator(generation_kwargs={"temperature": 0.7, "max_tokens": 500}))
        pipe.connect("prompt_builder", "generator")
        self.tag_pipe = pipe
    
    @component.output_types(documents=List[Document])
    def run(self, documents: List[Document]):
        for doc in documents:
            tags = self.tag_pipe.run({"prompt_builder": {"doc": doc}})
            doc.meta["metadata"]["tags"] = json.loads(tags["generator"]["replies"][0])
        
        return {"documents": documents}

    # @component.output_types(tags=list[list[str]])
    # def run(self, documents: List[Document]):
    #     tags = []
    #     for doc in documents:
    #         tag = self.tag_pipe.run({"prompt_builder": {"doc": doc}})
    #         tag = json.loads(tag["generator"]["replies"][0])
    #         tags.append(tag)
        
    #     return {"tags": tags}

In [15]:
prompt_template = """Given the context inside the triple backticks below, tag the topic. You may tag multiple topics.
                     Please format the tags as strings in a Python list, e.g. ["current events", "USA"].
                     Return only the list; return no other text.

                     Context:
                     ```
                     {{doc.content}}
                     ```

                     Tags as a Python list of strings:
                  """

llmtagger = SyncLLMTagger(prompt_template)

In [ ]:
doc1 = Document(content="Lebron James is the career points leader in the NBA.", meta={"metadata": {"tags": []}})
doc2 = Document(content="Stephen Curry is the only player in NBA history to be unanimously voted as the regular season MVP.", meta={"metadata": {"tags": []}})

documents = llmtagger.run([doc1, doc2])

In [18]:
for doc in documents["documents"]:
    print(doc)

Document(id=2a464df77d77ffdee5aed3442d97f0e6ee28b9cd1d97ed8e126de5592bd84e3d, content: 'Lebron James is the career points leader in the NBA.', meta: {'metadata': {'tags': ['sports', 'NBA', 'basketball', 'LeBron James']}})
Document(id=51b5288e898434688ba76c38c4a78d82990d36fbedbb42ca1ef24492ad00429d, content: 'Stephen Curry is the only player in NBA history to be unanimously voted as the regular season MVP.', meta: {'metadata': {'tags': ['sports', 'NBA', 'Stephen Curry', 'basketball', 'MVP']}})


## Upsert

In [12]:
from typing import Any, Dict, List, Optional

from haystack import Document, component, default_from_dict, default_to_dict, logging
from haystack.document_stores.types import DocumentStore, DuplicatePolicy
from haystack.utils import deserialize_document_store_in_init_params_inplace

@component
class DocumentUpserter(DocumentWriter):
    
    @component.output_types(documents_written=int)
    def run(self, documents: List[Document], policy: Optional[DuplicatePolicy] = None):
        """
        Run the DocumentWriter on the given input data.

        :param documents:
            A list of documents to write to the document store.
        :param policy:
            The policy to use when encountering duplicate documents.
        :returns:
            Number of documents written to the document store.

        :raises ValueError:
            If the specified document store is not found.
        """

        document_ids = [doc.id for doc in documents]
        self.document_store.delete_documents(document_ids)

        return_value = super(DocumentUpserter, self).run(documents, policy)
        return return_value

# @component
# class DocumentUpserter(DocumentWriter):
    
#     @component.output_types(documents_written=int)
#     def run(self, documents: List[Document], policy: Optional[DuplicatePolicy] = None):
#         """
#         Run the DocumentWriter on the given input data.

#         :param documents:
#             A list of documents to write to the document store.
#         :param policy:
#             The policy to use when encountering duplicate documents.
#         :returns:
#             Number of documents written to the document store.

#         :raises ValueError:
#             If the specified document store is not found.
#         """
#         if policy is None:
#             policy = self.policy
        
#         document_ids = [doc.id for doc in documents]
#         self.document_store.delete_documents(document_ids)
#         documents_written = self.document_store.write_documents(documents=documents, policy=policy)
#         return {"documents_written": documents_written}

In [18]:
DocumentUpserter.__mro__
# writer = DocumentUpserter(document_store=document_store)
# writer.__mro__

(__main__.DocumentUpserter,
 haystack.components.writers.document_writer.DocumentWriter,
 object)

In [5]:
from haystack import Pipeline, Document, component
from milvus_haystack import MilvusDocumentStore
from haystack.components.converters import PyPDFToDocument
from haystack.components.preprocessors import DocumentCleaner, DocumentSplitter
from haystack.components.embedders import SentenceTransformersDocumentEmbedder
from haystack.components.writers import DocumentWriter
from haystack.utils import Secret
from pymilvus import MilvusClient, DataType
from typing import List

In [8]:
doc1 = Document(id="king_james", content="Lebron James is the career points leader in the NBA.", meta={"metadata": {"tags": []}}, embedding=[0.1, 0.1])
doc2 = Document(id="night_night", content="Stephen Curry is the only player in NBA history to be unanimously voted as the regular season MVP.", meta={"metadata": {"tags": []}}, embedding=[0.1, 0.1])
documents = [doc1, doc2]

document_store = MilvusDocumentStore(
    collection_name = "TaggingTestCollection",
    collection_description = "Test collection",
    collection_properties = None,
    connection_args = {
        "uri": Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
        "token": Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value(),
        "secure": True
        },
    consistency_level = "Strong", # Strong, Bounded, Eventually, Session
    index_params = {
        "index_type": "AUTOINDEX",
        "metric_type": "COSINE",
    },
    search_params = {
        "params": {
            "level": 1
        }
    },
    drop_old = True,
)

writer = DocumentWriter(document_store)
writer.run(documents)

{'documents_written': 2}

In [21]:
prompt_template = """Given the context inside the triple backticks below, tag the topic. You may tag multiple topics.
                     Please format the tags as strings in a Python list, e.g. ["current events", "USA"].
                     Return only the list; return no other text.

                     Context:
                     ```
                     {{doc.content}}
                     ```

                     Tags as a Python list of strings:
                  """

tagger = SyncLLMTagger(prompt_template)

In [22]:
tagged_documents = tagger.run(documents)
tagged_documents["documents"]

[Document(id=king_james, content: 'Lebron James is the career points leader in the NBA.', meta: {'metadata': {'tags': ['sports', 'NBA', 'LeBron James', 'basketball']}}, embedding: vector of size 2),
 Document(id=night_night, content: 'Stephen Curry is the only player in NBA history to be unanimously voted as the regular season MVP.', meta: {'metadata': {'tags': ['sports', 'NBA', 'Stephen Curry', 'basketball', 'MVP']}}, embedding: vector of size 2)]

In [23]:
upserter = DocumentUpserter(document_store=document_store)

In [24]:
upserter.run(tagged_documents["documents"])

{'documents_written': 2}

## Full Pipeline

In [15]:
doc1 = Document(id="king_james", content="Lebron James is the career points leader in the NBA.", meta={"metadata": {"tags": []}}, embedding=[0.1, 0.1])
doc2 = Document(id="night_night", content="Stephen Curry is the only player in NBA history to be unanimously voted as the regular season MVP.", meta={"metadata": {"tags": []}}, embedding=[0.1, 0.1])
documents = [doc1, doc2]

document_store = MilvusDocumentStore(
    collection_name = "TaggingTestCollection",
    collection_description = "Test collection",
    collection_properties = None,
    connection_args = {
        "uri": Secret.from_env_var("ZILLIZ_CLUSTER_ENDPOINT").resolve_value(),
        "token": Secret.from_env_var("ZILLIZ_CLUSTER_TOKEN").resolve_value(),
        "secure": True
        },
    consistency_level = "Strong", # Strong, Bounded, Eventually, Session
    index_params = {
        "index_type": "AUTOINDEX",
        "metric_type": "COSINE",
    },
    search_params = {
        "params": {
            "level": 1
        }
    },
    drop_old = True,
)

writer = DocumentWriter(document_store)
writer.run(documents)

{'documents_written': 2}

In [16]:
pipe = Pipeline()

prompt_template = """Given the context inside the triple backticks below, tag the topic. You may tag multiple topics.
                     Please format the tags as strings in a Python list, e.g. ["current events", "USA"].
                     Return only the list; return no other text.

                     Context:
                     ```
                     {{doc.content}}
                     ```

                     Tags as a Python list of strings:
                  """

pipe.add_component("queryer", MilvusQueryRetriever())
pipe.add_component("tagger", SyncLLMTagger(prompt_template))
pipe.add_component("upserter", DocumentUpserter(document_store=document_store))
pipe.connect("queryer", "tagger")
pipe.connect("tagger", "upserter")

🚅 Components
  - queryer: MilvusQueryRetriever
  - tagger: SyncLLMTagger
  - upserter: DocumentUpserter
🛤️ Connections
  - queryer.documents -> tagger.documents (List[Document])
  - tagger.documents -> upserter.documents (List[Document])

In [19]:
# Tag all null
pipe.run({"queryer": {"collection_name": "TaggingTestCollection"}})

# Give IDs
# pipe.run({"queryer": {"collection_name": "TaggingTestCollection", "ids": ["night_night"]}})

{'upserter': {'documents_written': 1}}